# Nexel Chat API on Google Colab

This notebook runs the Nexel Chat FastAPI backend on Colab and exposes it with a temporary Cloudflare Tunnel URL.

Use this for demos and testing. Colab sessions sleep, disconnect, and reset, so this is not permanent production hosting.

## 1. Enable GPU

In Colab, open `Runtime > Change runtime type` and choose a GPU runtime before running the cells.

In [1]:
!nvidia-smi

Tue Apr 21 12:52:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone Nexel Chat and install dependencies

In [2]:
%cd /content
!rm -rf Nexel-Chat
!git clone https://github.com/NexelAi-Inc/Nexel-Chat.git
%cd /content/Nexel-Chat
!pip install -q -r requirements.txt

/content
Cloning into 'Nexel-Chat'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 91 (delta 30), reused 79 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 123.54 KiB | 1.18 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/Nexel-Chat
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 9.4 MB/s eta 0:00:

## 3. Configure the API

This uses the Hugging Face `transformers` backend. The first request may be slow while the model downloads and loads.

In [3]:
import os

os.environ["LLM_BACKEND"] = "transformers"
os.environ["LLM_MODE"] = "fast"
os.environ["HF_MODEL_ID"] = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
os.environ["HF_DEVICE_MAP"] = "auto"
os.environ["MAX_TOKENS"] = "768"
os.environ["TEMPERATURE"] = "0.4"
os.environ["MEMORY_DIR"] = "/content/nexel-chat-memory"
os.environ["CORS_ORIGINS"] = "https://nexelchat.netlify.app,https://nexelai.netlify.app,http://localhost:5173,http://127.0.0.1:5173"

print("Configured Nexel Chat API for Colab.")

Configured Nexel Chat API for Colab.


## 4. Download Cloudflare Tunnel

In [4]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
!./cloudflared --version

cloudflared version 2026.3.0 (built 2026-03-09-14:08 UTC)


## 5. Start API and public tunnel

When this cell prints `VITE_API_BASE_URL=...`, copy that value into Netlify for Nexel Chat and redeploy.

In [ ]:
import re
import subprocess
import time
from urllib.request import urlopen

api_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "agent.api:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

print("Starting Nexel Chat API...")
for _ in range(360):
    try:
        print(urlopen("http://127.0.0.1:8000/health", timeout=2).read().decode())
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("API did not become healthy. Check the API logs above.")

tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Starting Cloudflare tunnel...")
public_url = None
for line in iter(tunnel_process.stdout.readline, ""):
    print(line, end="")
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print("\nPaste this into Netlify for Nexel Chat:")
        print(f"VITE_API_BASE_URL={public_url}")
        print("VITE_FIREBASE_DATABASE_URL=https://nexel-ai-default-rtdb.firebaseio.com")
        print("VITE_NEXEL_AI_URL=https://nexelai.netlify.app")
        break

Starting Nexel Chat API...


## 6. Keep the notebook running

Leave the Colab tab open. If Colab disconnects or the runtime resets, the API URL stops working and you must run the notebook again, then update `VITE_API_BASE_URL` in Netlify with the new tunnel URL.